# 测试时覆盖依赖项¶
有些场景下，您可能需要在测试时覆盖依赖项。

即不希望运行原有依赖项（及其子依赖项）。

反之，要在测试期间（或只是为某些特定测试）提供只用于测试的依赖项，并使用此依赖项的值替换原有依赖项的值。


用例：外部服务¶
常见实例是调用外部第三方身份验证应用。

向第三方应用发送令牌，然后返回经验证的用户。

但第三方服务商处理每次请求都可能会收费，并且耗时通常也比调用写死的模拟测试用户更长。

一般只要测试一次外部验证应用就够了，不必每次测试都去调用。

此时，最好覆盖调用外部验证应用的依赖项，使用返回模拟测试用户的自定义依赖项就可以了。

## 使用 app.dependency_overrides 属性¶
对于这些用例，FastAPI 应用支持 app.dependency_overrides 属性，该属性就是字典。

要在测试时覆盖原有依赖项，这个字典的键应当是原依赖项（函数），值是覆盖依赖项（另一个函数）。

这样一来，FastAPI 就会调用覆盖依赖项，不再调用原依赖项。

In [ ]:
from typing import Annotated

from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient

app = FastAPI()


async def common_parameters(q: str | None = None, skip: int = 0, limit: int = 100):
    return {"q": q, "skip": skip, "limit": limit}


@app.get("/items/")
async def read_items(commons: Annotated[dict, Depends(common_parameters)]):
    return {"message": "Hello Items!", "params": commons}


@app.get("/users/")
async def read_users(commons: Annotated[dict, Depends(common_parameters)]):
    return {"message": "Hello Users!", "params": commons}


client = TestClient(app)


async def override_dependency(q: str | None = None):
    return {"q": q, "skip": 5, "limit": 10}


app.dependency_overrides[common_parameters] = override_dependency


def test_override_in_items():
    response = client.get("/items/")
    assert response.status_code == 200
    assert response.json() == {
        "message": "Hello Items!",
        "params": {"q": None, "skip": 5, "limit": 10},
    }


def test_override_in_items_with_q():
    response = client.get("/items/?q=foo")
    assert response.status_code == 200
    assert response.json() == {
        "message": "Hello Items!",
        "params": {"q": "foo", "skip": 5, "limit": 10},
    }


def test_override_in_items_with_params():
    response = client.get("/items/?q=foo&skip=100&limit=200")
    assert response.status_code == 200
    assert response.json() == {
        "message": "Hello Items!",
        "params": {"q": "foo", "skip": 5, "limit": 10},
    }

然后，使用 app.dependency_overrides 把覆盖依赖项重置为空字典：


`app.dependency_overrides = {}`